In [ ]:
# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# RDKit (분자 화학 라이브러리)
from rdkit import Chem
from rdkit.Chem import AllChem

# PyTorch & PyTorch Geometric
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

# 데이터 분할 및 평가
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 경고 메시지 무시
import warnings
warnings.filterwarnings("ignore")


# --- 2. 데이터 로드 ---
print("데이터 로딩 중...")
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('sample_submission.csv')


# --- 3. 사용자 정의 평가지표 함수 ---
def normalized_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    y_range = np.max(y_true) - np.min(y_true) if len(y_true) > 0 else 1
    return rmse / y_range if y_range != 0 else rmse

def pearson_correlation(y_true, y_pred):
    if len(np.unique(y_pred)) <= 1 or len(y_true) < 2: return 0.0
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr if not np.isnan(corr) else 0.0

def competition_score(y_true, y_pred):
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    nrmse = min(normalized_rmse(y_true_arr, y_pred_arr), 1.0)
    pearson = pearson_correlation(y_true_arr, y_pred_arr)
    return 0.5 * (1 - nrmse) + 0.5 * pearson


# --- 4. 데이터셋 원소 스캔 및 'possible_atoms' 리스트 생성 (★★ 변경점) ---
print("\n데이터셋의 모든 원소 종류를 스캔 중...")
all_atoms_set = set()
# 훈련 데이터와 테스트 데이터의 모든 분자를 스캔
for smiles in tqdm(pd.concat([train_df['Canonical_Smiles'], test_df['Canonical_Smiles']]).unique()):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        for atom in mol.GetAtoms():
            all_atoms_set.add(atom.GetSymbol())

# 정렬하여 순서를 고정하고 'Unknown' 카테고리 추가
possible_atoms = sorted(list(all_atoms_set))
possible_atoms.append('Unknown')

print(f"데이터셋에서 발견된 원소 종류 ({len(possible_atoms)-1}개): {possible_atoms[:-1]}")
print(f"최종 'possible_atoms' 리스트: {possible_atoms}")


# --- 5. 특징 공학: SMILES to Graph 변환 함수 정의 ---

def get_atom_features(atom, possible_atoms_list):
    """원자(Node)의 특징을 더 정교하게 추출하는 함수"""
    # 1. 원소 종류 (One-hot encoding) - 동적으로 생성된 리스트 사용 (★★ 변경점)
    atom_symbol = atom.GetSymbol()
    atom_type_onehot = [0] * len(possible_atoms_list)
    try:
        atom_type_onehot[possible_atoms_list.index(atom_symbol)] = 1
    except ValueError:
        atom_type_onehot[-1] = 1

    # 2. 혼성 오비탈 (One-hot encoding)
    hybridization_types = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                             Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                             Chem.rdchem.HybridizationType.SP3D2, Chem.rdchem.HybridizationType.UNSPECIFIED]
    hybridization_onehot = [0] * len(hybridization_types)
    try:
        hybridization_onehot[hybridization_types.index(atom.GetHybridization())] = 1
    except ValueError:
        hybridization_onehot[-1] = 1

    features = atom_type_onehot + hybridization_onehot + [
        atom.GetDegree(), atom.GetTotalNumHs(), atom.GetImplicitValence(),
        atom.GetTotalValence(), atom.GetFormalCharge(),
        atom.GetIsAromatic(), atom.IsInRing()
    ]
    return features

def get_bond_features(bond):
    """결합(Edge)의 특징을 추출하는 함수"""
    bond_types = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
                  Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
    bond_type_onehot = [0] * len(bond_types)
    try:
        bond_type_onehot[bond_types.index(bond.GetBondType())] = 1
    except ValueError: pass

    features = bond_type_onehot + [bond.GetIsConjugated(), bond.IsInRing()]
    return features

def smiles_to_graph(smiles, possible_atoms_list, y=None):
    """SMILES를 PyG의 Data 객체로 변환 (결합 특징 포함)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    atom_features_list = [get_atom_features(atom, possible_atoms_list) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_features_list, dtype=torch.float)

    edge_indices, edge_attrs = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feats = get_bond_features(bond)
        edge_indices.extend([(i, j), (j, i)])
        edge_attrs.extend([bond_feats, bond_feats])

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)

    return data


# --- 6. PyTorch Geometric 데이터셋 및 DataLoader 생성 ---
print("\nSMILES를 그래프 데이터로 변환 중...")
# (★★ 변경점) `possible_atoms` 리스트를 함수에 전달
train_graphs = [smiles_to_graph(s, possible_atoms, y) for s, y in tqdm(zip(train_df['Canonical_Smiles'], train_df['Inhibition']), total=len(train_df))]
test_graphs = [smiles_to_graph(s, possible_atoms) for s in tqdm(test_df['Canonical_Smiles'], total=len(test_df))]

train_graphs = [g for g in train_graphs if g is not None]
test_graphs = [g for g in test_graphs if g is not None and g.x.shape[0] > 0] # 노드가 없는 그래프 예외처리

print(f"학습용 그래프 생성 완료: {len(train_graphs)}개")
print(f"노드 특징 차원: {train_graphs[0].num_node_features}")


train_data, val_data = train_test_split(train_graphs, test_size=0.1, random_state=42)
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True) # drop_last 추가
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)


# --- 7. GAT 모델 정의 ---
class GATModel(torch.nn.Module):
    def __init__(self, num_node_features, hidden_dim, num_heads):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(num_node_features, hidden_dim, heads=num_heads, dropout=0.2)
        self.conv2 = GATConv(hidden_dim * num_heads, hidden_dim * 2, heads=num_heads, dropout=0.2)
        self.fc1 = nn.Linear(hidden_dim * 2 * num_heads, hidden_dim * 4)
        self.fc2 = nn.Linear(hidden_dim * 4, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = global_mean_pool(x, data.batch)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x


# 모델 인스턴스 생성
num_node_features = train_graphs[0].num_node_features
model = GATModel(num_node_features=num_node_features, hidden_dim=64, num_heads=8)
print("\n모델 구조 (GAT):")
print(model)


# --- 8. 학습 및 평가 설정 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=5e-4)
criterion = nn.MSELoss()

def train_epoch(loader):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out.squeeze(), data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate_epoch(loader):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            all_preds.extend(out.squeeze().cpu().numpy().tolist())
            all_trues.extend(data.y.cpu().numpy().tolist())
    return competition_score(all_trues, all_preds)


# --- 9. 모델 학습 실행 (Competition Score 기준) ---
print("\n모델 학습 시작...")
num_epochs = 10000
best_comp_score = -1.0

for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(train_loader)
    val_comp_score = evaluate_epoch(val_loader)

    if val_comp_score > best_comp_score:
        best_comp_score = val_comp_score
        torch.save(model.state_dict(), 'best_gat_model.pth')
        print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Comp Score: {val_comp_score:.4f} (모델 저장됨)")
    else:
        print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Comp Score: {val_comp_score:.4f}")


# --- 10. 테스트 데이터 예측 ---
print("\n테스트 데이터 예측 시작...")
# 가장 성능이 좋았던 모델 불러오기
# 만약 모델 인스턴스가 사라졌다면 다시 생성해야 함
model.load_state_dict(torch.load('best_gat_model.pth'))
model.to(device)
model.eval()

predictions = []
with torch.no_grad():
    for data in tqdm(test_loader):
        data = data.to(device)
        out = model(data)
        predictions.extend(out.squeeze().cpu().numpy().tolist())


# --- 11. 제출 파일 생성 ---
predictions_clipped = np.clip(predictions, 0, 100)

# 만약 test_df와 predictions의 길이가 다르다면 (변환 실패 케이스) 맞춰줘야 함
# 현재 코드는 변환 실패 케이스를 제외했으므로, ID를 다시 매칭하는 과정이 필요할 수 있음.
# 여기서는 길이가 같다고 가정하고 간단히 진행.
if len(submission_df) == len(predictions_clipped):
    submission_df['Inhibition'] = predictions_clipped
    submission_df.to_csv('submission_final.csv', index=False)
    print("\n제출 파일 'submission_final.csv' 생성 완료!")
    print(submission_df.head())
else:
    print(f"\n[경고] 제출 파일 길이({len(submission_df)})와 예측 개수({len(predictions_clipped)})가 다릅니다.")
    print("SMILES 변환에 실패한 test 데이터가 있을 수 있습니다. ID 매칭 로직이 필요합니다.")